# Tutorial: Multi-library MAPseq roadmap workflow

**Audience:** MAPseq analysts migrating multi-lane experiments from RStudio or earlier pymutscan releases.

**Prerequisites:** an editable pymutscan installation and the bundled synthetic data.

**Learning goals:** ingest named libraries without concatenation, retain library-resolved counts, use exact radius-two barcode correction, compare directional UMI logic, attach metadata, export a sparse matrix, and materialize non-causal template-switch evidence.


## Outline

1. Load the manifest and named preset.
2. Process two synthetic lanes.
3. Correct indices, barcodes, and UMIs.
4. Verify aggregate and library-resolved conservation.
5. Add sample metadata, sparse output, and shared-UMI evidence.
6. Exercise directional versus equal-score UMI behavior.


In [ ]:
from pathlib import Path
import shutil
import sqlite3

from pymutscan import (
    call_template_switch_evidence, collapse_database, collapse_umis,
    digest_libraries, export_sparse_matrix, get_preset,
    group_directional_sequences, group_similar_sequences,
    import_sample_metadata, load_library_manifest, map_sample_indices,
)

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
OUT = ROOT / 'notebooks/output/roadmap'
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)
DATABASE = OUT / 'multilane.sqlite'


## 1. Ingest named libraries

The manifest resolves paths relative to itself. `raw_counts` is the compatibility aggregate; `library_counts` retains lane identity.


In [ ]:
libraries = load_library_manifest(ROOT / 'examples/configs/two_lane_manifest.toml')
config = get_preset('mapseq-default')
ingestion = digest_libraries(libraries, DATABASE, config=config)
ingestion


## 2. Apply independent correction models

Indices are mapped against a whitelist. Barcodes use exact packed-key Hamming radius two. UMIs use the optional abundance-directional rule within each barcode/sample stratum.


In [ ]:
index_result = map_sample_indices(DATABASE, ['CGTGAT', 'ACATCG'])
barcode_result = collapse_database(DATABASE, collapse_max_dist=2)
umi_result = collapse_umis(DATABASE, method='directional')
{'indices': index_result, 'barcodes': barcode_result, 'umis': umi_result}


## 3. Check conservation and library provenance

All aggregate and library-resolved read totals must agree unless an explicit filter was requested.


In [ ]:
con = sqlite3.connect(DATABASE)
tables = ['raw_counts', 'collapsed_counts', 'umi_collapsed_counts', 'molecule_counts',
          'library_counts', 'library_collapsed_counts',
          'library_umi_collapsed_counts', 'library_molecule_counts']
totals = {table: con.execute(f'SELECT sum(read_count) FROM {table}').fetchone()[0] for table in tables}
library_rows = con.execute('SELECT library_id, lane FROM libraries ORDER BY library_id').fetchall()
con.close()
assert len(set(totals.values())) == 1
{'read_totals': totals, 'libraries': library_rows}


## 4. Metadata, sparse matrix, and shared-UMI evidence

The evidence caller reports support but does not claim that template switching caused a shared UMI.


In [ ]:
n_metadata = import_sample_metadata(DATABASE, ROOT / 'examples/synthetic/sample_metadata.tsv')
matrix = export_sparse_matrix(DATABASE, OUT / 'matrix')
switches = call_template_switch_evidence(DATABASE, min_reads_per_barcode=2)
{'metadata_rows': n_metadata, 'matrix': matrix, 'template_switch': switches}


## Exercise: when does directional correction differ?

Predict the representatives for counts 10, 6, and 2 along a one-base chain. Equal-score correction follows lexicographic greedy priority; directional correction requires $n_p \ge 2n_c-1$ at every traversed edge.


In [ ]:
umis = ['AAAA', 'AAAT', 'AATT']
counts = [10, 6, 2]
equal = group_similar_sequences(umis, [1, 1, 1], collapse_max_dist=1)
directional = group_directional_sequences(umis, counts, collapse_max_dist=1)
{'equal': equal, 'directional': directional}


## Pitfalls and extensions

- Library IDs must be globally unique before merging databases.
- Radius two is exact but performs 4,006 packed probes for a 30-base barcode; benchmark the intended scale.
- Directional UMI correction is an assay assumption, not a universally safer default.
- Shared UMIs are evidence compatible with several mechanisms, not causal labels.
- Extension: add biological columns to `sample_metadata.tsv` and rerun only the metadata/export cells.
